# QUOPS capability, one calculation at a time

This notebook follows single QUOPS widths through the direct and surface-code models, computes each number by hand, and checks it against the `capability` package. It then builds the capability curves for one hardware class and changes one modeling choice. The definitions are those of the methods note, `docs/quops_direct_and_surface.md`, and every choice is set in `capability/config.py`. The scripts in `scripts/` repeat sections 5 and 6 for all hardware classes and all choices. The notebook runs in about a minute.

In [1]:
import sys
from math import ceil, exp, log, log1p
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT/'capability').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from capability import curves, direct, quops, routing, surface
from capability.config import BASELINE, HARDWARE, N_MAX, vary
from capability.factories import FACTORIES

PASS = exp(-BASELINE.threshold)        # a width passes if the mean no-fault probability is at least 1/sqrt(e)
for key, hw in HARDWARE.items():
    print(f'{key}  {hw.name:20s} eps_1q = {hw.eps_1q:g}, eps_2q = {hw.eps_2q:g}')
print(f'pass threshold {PASS:.4f}')

H1  2D grid, uniform     eps_1q = 0.001, eps_2q = 0.001
H2  all-to-all, uniform  eps_1q = 0.001, eps_2q = 0.001
H3  2D grid, bimodal     eps_1q = 1e-06, eps_2q = 0.001
H4  all-to-all, bimodal  eps_1q = 1e-06, eps_2q = 0.001
pass threshold 0.6065


## 1. The QUOPS problem

A QUOPS circuit on an even number $w$ of qubits alternates rotation layers, an independent $R_P(\theta)$ on every qubit with $P$ uniform in $\{X, Y, Z\}$, and CNOT layers, an independent random pairing of the qubits. Every layer has size $w$. The shallow ensemble has $w$ layers and size $s = w^2$, and the deep ensemble has $w^2$ layers and size $s = w^3$. We start with a rotation layer, so $L$ layers give $L_R = \lceil L/2\rceil$ rotation layers and $L_C = \lfloor L/2\rfloor$ CNOT layers.

In [2]:
for w in (6, 12, 16):
    for ens in quops.ENSEMBLES:
        n_rot, n_cnot = quops.layer_counts(w, ens)
        print(f'w = {w:2d}, {ens:7s}: {quops.layers(w, ens):3d} layers ({n_rot} rotation, {n_cnot} CNOT), '
              f's = {quops.size(w, ens):,}')

w =  6, shallow:   6 layers (3 rotation, 3 CNOT), s = 36
w =  6, deep   :  36 layers (18 rotation, 18 CNOT), s = 216
w = 12, shallow:  12 layers (6 rotation, 6 CNOT), s = 144
w = 12, deep   : 144 layers (72 rotation, 72 CNOT), s = 1,728
w = 16, shallow:  16 layers (8 rotation, 8 CNOT), s = 256
w = 16, deep   : 256 layers (128 rotation, 128 CNOT), s = 4,096


## 2. Direct implementation on all-to-all hardware

Every native operation takes one time step and every location fails independently, so the no-fault probability of a circuit is the product of $1 - p_\ell$ over its locations. Preparation and measurement are $2w$ locations at $\varepsilon_{2Q}$. A CNOT layer on all-to-all hardware is $w/2$ CNOTs in one step, with every qubit busy. A rotation layer lasts as long as its slowest qubit, $D = 1$, 3, or 5 steps for $R_Z$, $R_X$, and $R_Y$ compiled into $H$ and $Z(\theta)$, and every qubit is a location at $\varepsilon_{1Q}$ in each of those steps, as a gate or as an idle. The average over the ensemble is

$$
\overline q = (1-\varepsilon_{2Q})^{2w + w L_C/2}\,\Big[\sum_{D} \Pr(D)\,(1-\varepsilon_{1Q})^{wD}\Big]^{L_R},
$$

where $\Pr(D=5) = 1-(2/3)^w$, $\Pr(D=3) = (2/3)^w-(1/3)^w$, and $\Pr(D=1) = (1/3)^w$.

In [3]:
def direct_all_to_all(w, ensemble, hw):
    n_rot, n_cnot = quops.layer_counts(w, ensemble)
    pD = {5: 1 - (2/3)**w, 3: (2/3)**w - (1/3)**w, 1: (1/3)**w}
    rotation_layer = sum(p*(1 - hw.eps_1q)**(w*D) for D, p in pD.items())
    return (1 - hw.eps_2q)**(2*w + n_cnot*w//2) * rotation_layer**n_rot


for key in ('H2', 'H4'):
    hw = HARDWARE[key]
    for ens in quops.ENSEMBLES:
        for w in (10, 12, 14):
            q = direct_all_to_all(w, ens, hw)
            q_pkg, _ = direct.mean_no_fault(w, ens, hw, BASELINE)
            assert abs(q - q_pkg) < 1e-12
            print(f"{key} {ens:7s} w = {w}: s = {quops.size(w, ens):5,}, q = {q:.3f}, -ln q = {-log(q):.3f}, "
                  f"{'passes' if q >= PASS else 'fails'}")

H2 shallow w = 10: s =   100, q = 0.746, -ln q = 0.293, passes
H2 shallow w = 12: s =   144, q = 0.658, -ln q = 0.419, passes
H2 shallow w = 14: s =   196, q = 0.567, -ln q = 0.567, fails
H2 deep    w = 10: s = 1,000, q = 0.064, -ln q = 2.754, fails
H2 deep    w = 12: s = 1,728, q = 0.009, -ln q = 4.765, fails
H2 deep    w = 14: s = 2,744, q = 0.001, -ln q = 7.568, fails
H4 shallow w = 10: s =   100, q = 0.956, -ln q = 0.045, passes
H4 shallow w = 12: s =   144, q = 0.941, -ln q = 0.060, passes
H4 shallow w = 14: s =   196, q = 0.925, -ln q = 0.078, passes
H4 deep    w = 10: s = 1,000, q = 0.761, -ln q = 0.273, passes
H4 deep    w = 12: s = 1,728, q = 0.631, -ln q = 0.461, passes
H4 deep    w = 14: s = 2,744, q = 0.486, -ln q = 0.721, fails


A rotation layer and a CNOT layer add $2w$ to the size. Under uniform noise (H2) they cost about $(5w + w/2)p$, and under bimodal noise (H4), where one-qubit gates and idles are nearly free, about $wp/2$. So $-\ln\overline q \approx 2.75\,ps$ and $0.25\,ps$, respectively, plus $2wp$ for preparation and measurement, for either ensemble. Before that last term, the criterion $-\ln \overline q \le 1/2$ allows $s \approx 180$ and $s \approx 2{,}000$. The capability is the largest size of either ensemble that passes: $144 = 12^2$ on H2, because the next deep size, $216 = 6^3$, fails, and $1{,}728 = 12^3$ on H4, because no even $w$ gives a passing shallow size above $40^2 = 1{,}600$. The cell below shows that the cost per unit of size is nearly the same for the two ensembles.

In [4]:
for key in ('H2', 'H4'):
    hw = HARDWARE[key]
    for ens in quops.ENSEMBLES:
        ws = range(2, 61, 2) if ens == 'shallow' else range(2, 17, 2)
        passing = [w for w in ws if direct_all_to_all(w, ens, hw) >= PASS]
        w = passing[-1]
        q = direct_all_to_all(w, ens, hw)
        print(f'{key} {ens:7s}: largest passing w = {w:2d}, s = {quops.size(w, ens):5,}, '
              f'-ln q / (p s) = {-log(q)/(hw.p*quops.size(w, ens)):.2f}')

H2 shallow: largest passing w = 12, s =   144, -ln q / (p s) = 2.91
H2 deep   : largest passing w =  4, s =    64, -ln q / (p s) = 2.67
H4 shallow: largest passing w = 40, s = 1,600, -ln q / (p s) = 0.30
H4 deep   : largest passing w = 12, s = 1,728, -ln q / (p s) = 0.27


## 3. Direct implementation on the 2D grid

On the grid the $w$ qubits fill an $r \times c$ rectangle with $rc = w$ and $c$ even. Before each CNOT layer, rounds of disjoint nearest-neighbor SWAPs, three CNOT steps each, bring every pair onto neighboring sites. The baseline router places the pairs on consecutive sites of a boustrophedon path and sorts along it by odd-even transposition. Here is one CNOT layer on $w = 8$. Each grid shows which qubit sits on each site.

In [5]:
w = 8
r, c = routing.grid_shape(w)
rng = np.random.default_rng(1)
pairs = quops.random_pairing(w, rng)
before = np.arange(r*c)                                    # site -> qubit
rounds, after = routing.ROUTERS['snake'](before, pairs, r, c)
assert routing.pairs_adjacent(after, pairs, c)
print(f'{r} x {c} grid; pairs to bring together: {[tuple(int(q) for q in p) for p in pairs]}')
print('before\n', before.reshape(r, c))
print('after\n', after.reshape(r, c))
n_swaps = sum(len(x) for x in rounds)
print(f'{len(rounds)} SWAP rounds, {n_swaps} SWAPs = {3*n_swaps} extra CNOT locations and {3*len(rounds)} extra time steps')

4 x 2 grid; pairs to bring together: [(5, 0), (1, 4), (2, 6), (3, 7)]
before
 [[0 1]
 [2 3]
 [4 5]
 [6 7]]
after
 [[0 5]
 [4 1]
 [3 7]
 [6 2]]
4 SWAP rounds, 7 SWAPs = 21 extra CNOT locations and 12 extra time steps


Every SWAP adds three CNOT locations, and every qubit not in a SWAP adds one idle location per step. The routing of a layer depends on where the previous layer left the qubits, so the average no-fault probability does not factorize, and the package averages over 64 sampled circuits. The grid lowers the no-fault probability at every width, and on H3 the capability falls from $1{,}728$ to $216 = 6^3$.

In [6]:
for w in (4, 6, 8):
    q_grid, err = direct.mean_no_fault(w, 'deep', HARDWARE['H3'], BASELINE)
    q_all = direct_all_to_all(w, 'deep', HARDWARE['H4'])
    print(f'deep, w = {w}: H3 (grid) q = {q_grid:.3f} +- {err:.3f}, H4 (all-to-all) q = {q_all:.3f}')

deep, w = 4: H3 (grid) q = 0.953 +- 0.001, H4 (all-to-all) q = 0.976
deep, w = 6: H3 (grid) q = 0.797 +- 0.002, H4 (all-to-all) q = 0.936
deep, w = 8: H3 (grid) q = 0.484 +- 0.003, H4 (all-to-all) q = 0.865


## 4. Counted sequential PBC at one width

We use H3, the deep ensemble, and $w=16$. Full PBC absorbs all Clifford gates, including CNOTs,
into Pauli axes. Each synthesized T consumes one state in a sequential measurement; the final
$w$ transformed measurements are also sequential. There is no CNOT runtime term.

With $r=\lceil\sqrt{w/2}\rceil$, $c=\lceil(w/2)/r\rceil$, the counted computational area is
$A=2w+2(r+c)+1$. For $F$ factories with $B$ outputs per batch, add $FB$ buffer tiles.
Starting cold, a batch takes $L=B\tau$ cycles and each measurement takes $d$ cycles.
We reconstruct the batch schedule below, independently of the package's schedule helper.

The loss is $(A+FB)C[-\ln(1-p_{\mathrm{SC}})]+N_T[-\ln(1-p_F)]+R\epsilon^2$.
These are effective resource-model assumptions, not a physical decoder simulation.


In [7]:
FIT = surface.FIT
CULTIVATION = FACTORIES[0]


def eps_for(nT):
    return 2.0**(-(nT - FIT['b'])/FIT['a'])


def sc_design(w, ensemble, hw, d, factory, F, nT, ch=BASELINE):
    R = w*quops.layer_counts(w, ensemble)[0]
    height = ceil(np.sqrt(w//2))
    columns = ceil((w//2)/height)
    batch = F*factory.states_per_batch
    tiles = 2*w+2*(height+columns)+1+batch
    service = ch.t_per_injection_steps*d
    waves, last = divmod(R*nT-1, batch)
    cycles = factory.batch_cycles + waves*max(factory.batch_cycles, batch*service) + (last+1)*service + w*d
    loss = dict(patches=tiles*cycles*-log1p(-surface.p_sc(d, hw, ch)),
                magic=R*nT*-log1p(-factory.error), synthesis=R*eps_for(nT)**2)
    runs = min(ch.executions, int(7*24*3600/(cycles*ch.cycle_seconds)))
    copies = ceil(ch.executions/max(runs, 1))
    return dict(qubits=copies*(tiles*2*d*d+F*factory.qubits), cycles=cycles, loss=loss,
                passes=sum(loss.values()) <= ch.threshold and runs >= 1)


hw = HARDWARE['H3']
x = surface.min_qubits(16, 'deep', hw, BASELINE)
f = next(f for f in FACTORIES if f.name == x['factory'])
y = sc_design(16, 'deep', hw, x['distance'], f, x['factories'], x['T_per_rotation'])
print(f"package: {x['qubits']:,.0f} qubits, d={x['distance']}, F={x['factories']}, nT={x['T_per_rotation']}")
print(f"by hand: {y['qubits']:,} qubits, {y['cycles']:,.0f} cycles")
print('loss:', y['loss'])
assert y['qubits'] == x['qubits'] and y['cycles'] == x['cycles']
for key, value in y['loss'].items():
    assert np.isclose(value, x['loss_'+key], rtol=1e-12)
assert x['cnot_cycles'] == 0


package: 13,422 qubits, d=11, F=2, nT=22
by hand: 13,422 qubits, 2,252,998 cycles
loss: {'patches': 0.2788896293384554, 'magic': 9.0112000090112e-05, 'synthesis': 0.16621603654869127}


Factory supply can dominate a sequential schedule. More factories help only until the data block
can consume a state every $d$ cycles. Beyond that point they add area without reducing the runtime.
The following cell searches a bounded cultivation-only menu at each distance.


In [8]:
print(' d  factories  n_T  qubits  loss (patches, magic, synthesis)')
for d in (9, 11, 13):
    best = None
    for F in range(1, 65):
        for nT in range(10, 41):
            z = sc_design(16, 'deep', hw, d, CULTIVATION, F, nT)
            if z['passes'] and (best is None or z['qubits'] < best[2]['qubits']):
                best = (F, nT, z)
    if best is None:
        print(f'{d:2d}: no passing design in this menu')
        continue
    F, nT, z = best
    print(f"{d:2d}  {F:9d}  {nT:3d}  {z['qubits']:6,}  " + ', '.join(f'{v:.3f}' for v in z['loss'].values()))

 d  factories  n_T  qubits  loss (patches, magic, synthesis)
 9: no passing design in this menu
11          2   22  13,422  0.279, 0.000, 0.166


13          1   20  16,572  0.033, 0.000, 0.409


Increasing distance reduces patch-cycle errors but increases area. The package searches the full
factory menu, T counts 5 through 160, odd distances 3 through 99, and factory counts through serial
consumption saturation. Copy counts satisfy the discrete one-week execution constraint.

## 5. Capability curves and the crossover

The capability $C(N)$ is the largest passing size that fits in $N$ physical qubits. The crossover
is the first qubit budget at which PBC exceeds direct execution. The curves stop at $N=10^6$.


In [9]:
d_rows = direct.sweep(hw, BASELINE)
s_rows = surface.sweep(hw, BASELINE)
N_x = curves.crossover(d_rows, s_rows)

fig, ax = plt.subplots(figsize=(6, 4))
for rows, color, label in ((d_rows, '#2a78d6', 'Direct'), (s_rows, '#eb6834', 'Surface code')):
    y = curves.curve(rows).astype(float)
    y[y == 0] = np.nan
    ax.step(curves.N_GRID, y, where='post', color=color, lw=2, label=label)
ax.plot([6, x['qubits']], [216, 4096], 'o', color='#2b2b2b', ms=5, label='Sections 3 and 4')
ax.axvline(N_x, color='#8a8a8a', lw=1, ls=':')
ax.text(N_x*1.3, 3e8, f'crossover\n{N_x:,.0f} qubits', fontsize=8, va='top')
ax.set(xscale='log', yscale='log', xlim=(1, N_MAX), ylim=(1, 1e9), xlabel='Physical qubits N',
       ylabel='QUOPS capability s', title='H3: 2D grid, bimodal')
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
ax.grid(alpha=.15)
ax.legend(frameon=False, fontsize=9, loc='upper left')
plt.show()

/tmp/ipykernel_2988130/80659956.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The highlighted points are the direct design of Section 3 and the counted sequential PBC design
of Section 4. The cell below computes the current defaults for all four hardware classes.


In [10]:
for key, hw_k in HARDWARE.items():
    d_k, s_k = direct.sweep(hw_k, BASELINE), surface.sweep(hw_k, BASELINE)
    print(f'{key}: direct s = {curves.capability(d_k, N_MAX):5,}, crossover N = {curves.crossover(d_k, s_k):6,.0f}, '
          f'surface-code s at 1e6 = {surface.capability(N_MAX, hw_k, BASELINE, s_k):.2e}')

H1: direct s =    64, crossover N =  6,348, surface-code s at 1e6 = 4.59e+07


H2: direct s =   144, crossover N =  6,348, surface-code s at 1e6 = 4.59e+07


H3: direct s =   216, crossover N =  6,422, surface-code s at 1e6 = 1.24e+08


H4: direct s = 1,728, crossover N = 11,188, surface-code s at 1e6 = 1.24e+08


## 6. Changing one choice

The measurement duration can be changed with `vary`. Two surgery steps per T consumption increase
service time from $d$ to $2d$ cycles while keeping the same counted geometry and supply policy.


In [11]:
slower = vary(t_per_injection_steps=2)
s_slower = surface.sweep(hw, slower)
print(f'H3 crossover: baseline {N_x:,.0f}, two steps per T {curves.crossover(d_rows, s_slower):,.0f}')


H3 crossover: baseline 6,422, two steps per T 6,422


With cultivation factories only, the output error of the factory caps the capability. A deep-ensemble circuit has about $s/2$ rotations of $n_T$ $T$ gates, so the factory term of the loss alone requires $s \lesssim 1/(n_T P_{\rm factory})$, whatever the number of qubits.

In [12]:
cultivation_only = vary(factories=('cultivation',))
s_cult = surface.sweep(hw, cultivation_only)
z = surface.best_at(N_MAX, hw, cultivation_only, s_cult)
print(f"cultivation only, at most 1e6 qubits: s = {z['s']:.3g} on {z['qubits']:,.0f} qubits, "
      f"n_T = {z['T_per_rotation']}, factory share of the loss {z['loss_magic']:.3f}")
print(f"ceiling 1/(n_T P_factory) = {1/(z['T_per_rotation']*CULTIVATION.error):.3g}")

cultivation only, at most 1e6 qubits: s = 1.06e+07 on 433,630 qubits, n_T = 44, factory share of the loss 0.469
ceiling 1/(n_T P_factory) = 1.14e+07
